# StellarSeeker - Tutorial Completo

## Análise de Curvas de Luz e Descoberta de Exoplanetas com Dados do TESS

Este Jupyter Notebook fornece um tutorial completo sobre como usar o StellarSeeker para analisar dados astronômicos reais da missão TESS e descobrir exoplanetas.

### Objetivos de Aprendizado

Ao final deste tutorial, você será capaz de:
- Acessar dados profissionais do telescópio espacial TESS
- Criar e interpretar curvas de luz estelares
- Calcular e analisar periodogramas
- Realizar dobradura de fase (phase folding)
- Detectar automaticamente trânsitos planetários
- Contribuir para projetos de ciência cidadã como Planet Hunters TESS

## 1. Introdução ao Método de Trânsitos

### Como Funciona a Detecção de Exoplanetas por Trânsito

O método de trânsitos é uma das técnicas mais bem-sucedidas para detectar exoplanetas. Quando um planeta orbita sua estrela hospedeira e passa diretamente entre a estrela e o observador (neste caso, o telescópio TESS), ele bloqueia uma pequena fração da luz estelar.

![Diagrama do Método de Trânsito](https://exoplanets.nasa.gov/internal_resources/2056/)

**Pontos-chave:**
- A redução no brilho é proporcional à área do planeta: `(R_planeta / R_estrela)²`
- Para um planeta do tamanho da Terra orbitando uma estrela como o Sol, a redução é de apenas ~0.008%
- Para um Júpiter quente, a redução pode chegar a 1%
- O período entre trânsitos revela o período orbital do planeta
- A duração do trânsito informa sobre a distância do planeta à estrela

## 2. Configurando o Ambiente

Primeiro, vamos importar as bibliotecas necessárias:

In [ ]:
# Importar bibliotecas
import numpy as np
import matplotlib.pyplot as plt

# Importar StellarSeeker
from stellar_seeker import StellarSeeker
from stellar_seeker.utils import tic_to_coordinates, btjd_to_datetime, format_period

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Ambiente configurado com sucesso! 🚀")

## 3. Escolhendo um Alvo

Vamos analisar **π Mensae** (TIC 261136679), uma estrela conhecida por ter um exoplaneta confirmado.

### Sobre π Mensae:
- **Tipo espectral**: G1V (similar ao Sol)
- **Distância**: ~59.7 anos-luz
- **Exoplaneta**: π Mensae b, descoberto pelo TESS em 2018
- **Período orbital**: ~6.26 dias
- **Tipo de planeta**: Super-Terra ou Mini-Netuno

In [ ]:
# Inicializar o StellarSeeker com o TIC de π Mensae
tic_id = 261136679
seeker = StellarSeeker(tic_id=tic_id)

print(f"StellarSeeker inicializado para TIC {tic_id}")

# Obter coordenadas da estrela
coords = tic_to_coordinates(tic_id)
if coords:
    ra, dec = coords
    print(f"Coordenadas: RA = {ra:.4f}°, Dec = {dec:.4f}°")

## 4. Buscando Dados Disponíveis

O TESS observa o céu em setores de aproximadamente 27 dias cada. Vamos verificar quais setores têm dados para nossa estrela.

In [ ]:
# Buscar setores disponíveis
sectors = seeker.search_sectors()

print(f"\n📊 Setores TESS disponíveis para TIC {tic_id}:\n")

for i, sector in enumerate(sectors):
    print(f"[{i}] Setor {sector['sector']}: {sector['duration']:.1f} dias")

if sectors:
    print(f"\n✅ Total: {len(sectors)} setores encontrados")

## 5. Baixando a Curva de Luz

Agora vamos baixar os dados do primeiro setor disponível. A curva de luz contém:
- **Tempo (BTJD)**: Barycentric TESS Julian Date
- **Fluxo**: Contagem de elétrons por segundo
- **Erro de fluxo**: Incerteza na medição
- **Qualidade**: Flag indicando problemas nos dados

In [ ]:
# Baixar dados do primeiro setor
lc = seeker.download_lightcurve(sector_index=0)

print(f"✅ Download concluído!")
print(f"   Pontos de dados: {len(lc)}")
print(f"   Duração: {lc.time.max() - lc.time.min():.1f} dias")
print(f"   Fluxo médio: {np.median(lc.flux.value):.2f} e⁻/s")

## 6. Visualizando a Curva de Luz

Vamos criar nosso primeiro gráfico da curva de luz. A normalização divide todos os valores pela mediana, facilitando a visualização de variações relativas.

In [ ]:
# Plotar curva de luz
fig = seeker.plot_lightcurve(normalize=True, color='darkblue', marker='.')
plt.show()

### Interpretando a Curva de Luz

Observe no gráfico:
1. **Eixo X**: Tempo em BTJD (Barycentric TESS Julian Date)
2. **Eixo Y**: Fluxo normalizado (1.0 = brilho médio)
3. **Dispersão**: Variações devido a ruído instrumental e variabilidade estelar
4. **Quedas**: Possíveis indicadores de trânsitos planetários ou eclipses

**Pergunta**: Você consegue identificar alguma queda periódica no brilho?

## 7. Calculando o Periodograma

Para encontrar periodicidades nos dados, usamos o **Periodograma de Lomb-Scargle**. Esta técnica é ideal para dados astronômicos porque:
- Funciona com dados irregularmente espaçados
- É sensível a sinais periódicos fracos
- Fornece uma medida estatística de significância

O pico mais alto no periodograma indica o período mais provável do sinal.

In [ ]:
# Calcular periodograma
pg = seeker.calculate_periodogram(
    oversample_factor=10,
    minimum_period=0.5,  # dias
    maximum_period=15    # dias
)

# Obter período candidato
period = seeker.get_primary_period()

print(f"🎯 Período de máxima potência: {period:.6f} dias")
print(f"   Em formato legível: {format_period(period)}")

# Comparar com valor esperado
expected_period = 6.26
diff_percent = abs(period - expected_period) / expected_period * 100
print(f"\n📋 Comparação com valor conhecido:")
print(f"   Valor esperado: ~{expected_period} dias")
print(f"   Diferença: {diff_percent:.2f}%")

In [ ]:
# Visualizar periodograma
fig = seeker._plot_periodogram()
plt.show()

### Interpretando o Periodograma

- **Eixo X**: Períodos testados (em dias)
- **Eixo Y**: Potência do sinal (quanto maior, mais significativo)
- **Linha vermelha**: Período de máxima potência (candidato principal)

**Atenção**: Múltiplos picos podem indicar:
- Harmônicos (múltiplos do período real)
- Aliasing (artefatos de amostragem)
- Múltiplos planetas ou estrelas binárias

## 8. Dobradura de Fase (Phase Folding)

A dobradura de fase é uma técnica poderosa que "empilha" múltiplos ciclos da curva de luz, alinhando-os pelo período calculado. Isso:
- Amplifica sinais periódicos fracos
- Facilita a visualização de trânsitos
- Permite medir parâmetros do trânsito com precisão

In [ ]:
# Realizar phase folding
fig, phase, flux = seeker.fold_phase(wrap_phase=2)
plt.show()

print("✅ Phase folding concluído!")
print(f"   Fase: {len(phase)} pontos")
print(f"   Período usado: {period:.6f} dias")

### Interpretando a Curva Dobrada

Na curva dobrada em fase:
- **Fase 0 e 1**: Momento do trânsito (meio do evento)
- **Forma em U ou V**: Característica de trânsitos planetários
- **Profundidade**: Relacionada ao tamanho do planeta
- **Duração**: Informa sobre a órbita do planeta

Se o período estiver correto, os pontos se alinham formando um padrão claro!

## 9. Detectando Trânsitos Automaticamente

O StellarSeeker inclui um algoritmo de detecção automática de trânsitos que identifica quedas significativas no fluxo estelar.

In [ ]:
# Detectar trânsitos
transits = seeker.detect_transits(min_depth=0.0005, min_duration=0.01)

print(f"🔍 Trânsitos detectados: {len(transits)}\n")

if transits:
    print("Principais candidatos:")
    print("-" * 70)
    
    for i, t in enumerate(transits[:5], 1):
        dt = btjd_to_datetime(t['time_center'])
        print(f"\n{i}. Data: {dt.strftime('%Y-%m-%d %H:%M UTC')}")
        print(f"   BTJD: {t['time_center']:.4f}")
        print(f"   Profundidade: {t['depth']:.5f} ({t['depth']*100:.3f}%)")
        print(f"   Duração: {t['duration']:.4f} dias ({t['duration']*24:.2f} horas)")
        print(f"   SNR: {t['snr']:.1f}")

### Critérios para um Bom Candidato

| Parâmetro | Bom Candidato | Comentário |
|-----------|---------------|------------|
| **SNR** | > 7 | Relação sinal-ruído alta |
| **Profundidade** | 0.001 - 0.01 | Típico para exoplanetas |
| **Duração** | 2-8 horas | Depende do período |
| **Formato** | U ou V | Trânsito limpo |
| **Repetição** | Múltiplos eventos | Confirma periodicidade |

## 10. Exportando Resultados

Vamos salvar nossos resultados em múltiplos formatos para análise posterior e compartilhamento.

In [ ]:
# Exportar resultados
results = seeker.export_results(
    output_dir='./results_pi_mensae',
    formats=['png', 'csv', 'fits', 'json']
)

print("💾 Resultados exportados:\n")
for fmt, path in results.items():
    print(f"  ✓ {path}")

## 11. Explorando Outros Alvos

Agora que você dominou o básico, tente analisar outras estrelas interessantes:

In [ ]:
# Lista de alvos interessantes para explorar
alvos = [
    {
        'nome': 'Algol (Beta Persei)',
        'tic': 346783960,
        'tipo': 'Binária eclipsante',
        'periodo': '~2.87 dias',
        'descricao': 'Uma das primeiras variáveis eclipsantes descobertas'
    },
    {
        'nome': 'TYC 7037-89-1',
        'tic': 168789840,
        'tipo': 'Sistema sextuplo',
        'periodo': '~1.57 dias',
        'descricao': 'Sistema raro com seis estrelas'
    },
    {
        'nome': 'TOI-700',
        'tic': 150428135,
        'tipo': 'Estrela com planeta na zona habitável',
        'periodo': '~37 dias',
        'descricao': 'Planeta do tamanho da Terra na zona habitável'
    }
]

print("🌟 Alvos sugeridos para exploração:\n")
print("=" * 70)

for alvo in alvos:
    print(f"\n📍 {alvo['nome']} (TIC {alvo['tic']})")
    print(f"   Tipo: {alvo['tipo']}")
    print(f"   Período: {alvo['periodo']}")
    print(f"   {alvo['descricao']}")

## 12. Contribuindo para a Ciência Cidadã

### Projetos onde você pode contribuir:

1. **[Planet Hunters TESS](https://www.planethunters.org/tess)**
   - Classificação visual de curvas de luz
   - Identificação de candidatos não detectados por algoritmos

2. **[Zooniverse](https://www.zooniverse.org/)**
   - Plataforma com múltiplos projetos de astronomia
   - Não requer conhecimento especializado

3. **[Exoplanet Explorers](https://www.exoplanetexplorers.org/)**
   - Foco em dados do Kepler/K2
   - Comunidade ativa de discussão

### Dicas para Contribuições Efetivas:

- Documente suas descobertas cuidadosamente
- Compare com catálogos existentes (NASA Exoplanet Archive)
- Participe de fóruns de discussão
- Colabore com outros cidadãos cientistas

## 13. Resumo e Próximos Passos

### O que aprendemos:

✅ Como acessar dados profissionais do TESS  
✅ Como criar e interpretar curvas de luz  
✅ Como calcular e analisar periodogramas  
✅ Como realizar phase folding  
✅ Como detectar trânsitos automaticamente  
✅ Como exportar resultados para análise posterior  

### Próximos passos:

1. **Explore mais alvos**: Use a lista acima ou encontre seus próprios TICs
2. **Aprofunde-se na teoria**: Estude mecânica orbital e fotometria
3. **Junte-se à comunidade**: Participe de fóruns e projetos de ciência cidadã
4. **Contribua com código**: Melhorias são bem-vindas no StellarSeeker!

### Recursos Adicionais:

- [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/)
- [TESS Mission Website](https://tess.mit.edu/)
- [LightKurve Documentation](https://docs.lightkurve.org/)
- [Astropy Tutorials](https://learn.astropy.org/)

## Apêndice: Código para Análise Personalizada

Aqui estão alguns snippets úteis para análises avançadas:

In [ ]:
# Exemplo: Analisar múltiplos setores combinados
def analyze_multiple_sectors(tic_id, sector_indices=None):
    """Analisa múltiplos setores de uma vez."""
    seeker = StellarSeeker(tic_id=tic_id)
    sectors = seeker.search_sectors()
    
    if sector_indices is None:
        sector_indices = range(min(3, len(sectors)))
    
    print(f"Analisando {len(sector_indices)} setores para TIC {tic_id}\n")
    
    for idx in sector_indices:
        print(f"\n--- Setor {idx} ---")
        try:
            lc = seeker.download_lightcurve(sector_index=idx)
            pg = seeker.calculate_periodogram()
            period = seeker.get_primary_period()
            print(f"Período: {period:.4f} dias")
        except Exception as e:
            print(f"Erro: {e}")
    
    return seeker

# Uso:
# seeker = analyze_multiple_sectors(261136679)

In [ ]:
# Exemplo: Estimar raio do planeta
def estimate_planet_radius(transit_depth, stellar_radius_solar=1.0):
    """
    Estima o raio de um exoplaneta baseado na profundidade do trânsito.
    
    Args:
        transit_depth: Profundidade do trânsito (fração do fluxo)
        stellar_radius_solar: Raio da estrela em unidades solares
    
    Returns:
        Raio do planeta em raios terrestres
    """
    R_sun = 696340  # km
    R_earth = 6371  # km
    
    # (R_planet / R_star)^2 = transit_depth
    radius_ratio = np.sqrt(transit_depth)
    
    R_planet_km = radius_ratio * stellar_radius_solar * R_sun
    R_planet_earth = R_planet_km / R_earth
    
    return R_planet_earth

# Exemplo para π Mensae b
if transits:
    depth = transits[0]['depth']
    radius = estimate_planet_radius(depth, stellar_radius_solar=1.1)
    print(f"Raio estimado do planeta: {radius:.2f} R_earth")
    
    if radius < 1.25:
        tipo = "Terra"
    elif radius < 2:
        tipo = "Super-Terra"
    elif radius < 4:
        tipo = "Mini-Netuno"
    else:
        tipo = "Gigante"
    
    print(f"Classificação provável: {tipo}")

---

**StellarSeeker** - Democratizando a descoberta de exoplanetas 🌍🔭✨

*Desenvolvido com ❤️ para a comunidade de astronomia amadora e cidadãos cientistas*